### RAG Pipelines - Data ingestion to Vector DB Pipeline


In [15]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path



In [16]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Ignoring wrong pointing object 131 0 (offset 0)
Ignoring wrong pointing object 567 0 (offset 0)


Found 4 PDF files to process

Processing: dsa.pdf
  ✓ Loaded 841 pages

Processing: multi-page-pdf-a4-size.pdf
  ✓ Loaded 20 pages

Processing: text-only.pdf
  ✓ Loaded 5 pages

Processing: with-images.pdf
  ✓ Loaded 6 pages

Total documents loaded: 872


In [17]:
all_pdf_documents

[Document(metadata={'producer': 'macOS Version 11.2.3 (Build 20D91) Quartz PDFContext', 'creator': 'Keynote', 'creationdate': "D:20210531125452Z00'00'", 'title': 'Slides in PDF', 'moddate': "D:20210531125452Z00'00'", 'source': '..\\data\\pdf_files\\dsa.pdf', 'total_pages': 841, 'page': 0, 'page_label': '1', 'source_file': 'dsa.pdf', 'file_type': 'pdf'}, page_content='Elshad Karimov from AppMillers\nJava Data Structures and \nAlgorithms Masterclass\nLearning Data Structures and Algorithms'),
 Document(metadata={'producer': 'macOS Version 11.2.3 (Build 20D91) Quartz PDFContext', 'creator': 'Keynote', 'creationdate': "D:20210531125452Z00'00'", 'title': 'Slides in PDF', 'moddate': "D:20210531125452Z00'00'", 'source': '..\\data\\pdf_files\\dsa.pdf', 'total_pages': 841, 'page': 1, 'page_label': '2', 'source_file': 'dsa.pdf', 'file_type': 'pdf'}, page_content='Elshad Karimov from AppMillers\nJava Data Structures and Algorithms \nMasterclass\n43 Sections\n400+ Lectures\n45+ hours of Content\n1

In [18]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [19]:
chunks=split_documents(all_pdf_documents)
chunks

Split 872 documents into 899 chunks

Example chunk:
Content: Elshad Karimov from AppMillers
Java Data Structures and 
Algorithms Masterclass
Learning Data Structures and Algorithms...
Metadata: {'producer': 'macOS Version 11.2.3 (Build 20D91) Quartz PDFContext', 'creator': 'Keynote', 'creationdate': "D:20210531125452Z00'00'", 'title': 'Slides in PDF', 'moddate': "D:20210531125452Z00'00'", 'source': '..\\data\\pdf_files\\dsa.pdf', 'total_pages': 841, 'page': 0, 'page_label': '1', 'source_file': 'dsa.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'macOS Version 11.2.3 (Build 20D91) Quartz PDFContext', 'creator': 'Keynote', 'creationdate': "D:20210531125452Z00'00'", 'title': 'Slides in PDF', 'moddate': "D:20210531125452Z00'00'", 'source': '..\\data\\pdf_files\\dsa.pdf', 'total_pages': 841, 'page': 0, 'page_label': '1', 'source_file': 'dsa.pdf', 'file_type': 'pdf'}, page_content='Elshad Karimov from AppMillers\nJava Data Structures and \nAlgorithms Masterclass\nLearning Data Structures and Algorithms'),
 Document(metadata={'producer': 'macOS Version 11.2.3 (Build 20D91) Quartz PDFContext', 'creator': 'Keynote', 'creationdate': "D:20210531125452Z00'00'", 'title': 'Slides in PDF', 'moddate': "D:20210531125452Z00'00'", 'source': '..\\data\\pdf_files\\dsa.pdf', 'total_pages': 841, 'page': 1, 'page_label': '2', 'source_file': 'dsa.pdf', 'file_type': 'pdf'}, page_content='Elshad Karimov from AppMillers\nJava Data Structures and Algorithms \nMasterclass\n43 Sections\n400+ Lectures\n45+ hours of Content\n1

### Embedding & VectorStoreDB

In [20]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [21]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8063.56it/s]


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [22]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 4495


In [23]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store into the vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 899 texts...


Batches: 100%|██████████| 29/29 [00:14<00:00,  1.97it/s]


Generated embeddings with shape: (899, 384)
Adding 899 documents to vector store...
Successfully added 899 documents to vector store
Total documents in collection: 5394


### Retriever Pipeline From VectorStore

In [24]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [25]:
rag_retriever.retrieve("What are different types of algorithms?")

Retrieving documents for query: 'What are different types of algorithms?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 60.70it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_c1d9cd63_25',
  'content': 'Types of Algorithms',
  'metadata': {'creationdate': "D:20210531125452Z00'00'",
   'creator': 'Keynote',
   'moddate': "D:20210531125452Z00'00'",
   'doc_index': 25,
   'source_file': 'dsa.pdf',
   'page_label': '26',
   'title': 'Slides in PDF',
   'content_length': 19,
   'file_type': 'pdf',
   'page': 25,
   'source': '..\\data\\pdf_files\\dsa.pdf',
   'producer': 'macOS Version 11.2.3 (Build 20D91) Quartz PDFContext',
   'total_pages': 841},
  'similarity_score': 0.8811613991856575,
  'distance': 0.1188386008143425,
  'rank': 1},
 {'id': 'doc_80b69a4a_25',
  'content': 'Types of Algorithms',
  'metadata': {'creator': 'Keynote',
   'doc_index': 25,
   'source': '..\\data\\pdf_files\\dsa.pdf',
   'producer': 'macOS Version 11.2.3 (Build 20D91) Quartz PDFContext',
   'creationdate': "D:20210531125452Z00'00'",
   'moddate': "D:20210531125452Z00'00'",
   'page': 25,
   'total_pages': 841,
   'source_file': 'dsa.pdf',
   'title': 'Slides in PDF',


### Integrate VectorDb Context pipeline With LLM output

In [ ]:
### Simple RAG pipeline with Groq LLM

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY") or ""

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content



In [34]:
answer=rag_simple("What are different types of algorithms?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What are different types of algorithms?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 74.19it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


There are several types of algorithms, including:

1. **Sorting Algorithms**: Used to arrange data in a specific order (e.g., Bubble Sort, Quick Sort).
2. **Searching Algorithms**: Used to find a specific item in a dataset (e.g., Linear Search, Binary Search).
3. **Graph Algorithms**: Used to solve problems involving graphs (e.g., Dijkstra's Algorithm, Bellman-Ford Algorithm).
4. **Dynamic Programming Algorithms**: Used to solve problems by breaking them down into smaller sub-problems (e.g., Fibonacci Series, Longest Common Subsequence).
5. **Greedy Algorithms**: Used to solve problems by making the locally optimal choice (e.g., Huffman Coding, Activity Selection Problem).
6. **Backtracking Algorithms**: Used to solve problems by trying all possible solutions (e.g., N-Queens Problem, Sudoku).
7. **Divide and Conquer Algorithms**: Used to solve problems by dividing them into smaller sub-problems (e.g., Merge Sort, Fast Fourier Transform).
8. **Linear Algebra Algorithms**: Used to solve 